# Performance Optimisation Guide I = Batch Matmul + GPU case

## Input:

In [1]:
module {
  func.func @matmul(%arg0: tensor<8x512x1024xf32>, %arg1: tensor<8x1024x512xf32>) -> tensor<8x512x512xf32> {
    %0 = tosa.matmul %arg0, %arg1 : (tensor<8x512x1024xf32>, tensor<8x1024x512xf32>) -> tensor<8x512x512xf32>
    return %0 : tensor<8x512x512xf32>
  }
}

!../../build/bin/ragdoll-opt \
    ../operator-cases/matmul.mlir \
    --pass-pipeline="builtin.module(func.func(tosa-to-linalg-named))" > ../../build/.matmul.mlir.linalg

!nohup iree-compile ../../build/.matmul.mlir.linalg \
-o ../../build/.matmul.candidate1.vmfb \
--mlir-print-ir-after-all \
-mlir-print-ir-after-change \
--td-matmul-strategy-use-wmma \
--iree-codegen-llvmgpu-enable-transform-dialect-matmul-tensorcore-strategy \
--iree-codegen-llvmgpu-enable-transform-dialect-aligned-matmul \
--iree-flow-fuse-multi-use \
--iree-flow-enable-fuse-padding-into-linalg-consumer-ops   \
--iree-flow-enable-fuse-padding-into-linalg-producer-ops  \
--iree-hal-target-backends=cuda \
--iree-hal-cuda-llvm-target-arch=sm_70 > ../../build/dump.mlir 2>&1

!nohup iree-run-module \
    --device=cuda \
    --module="../../build/.matmul.vmfb" \
    --function=matmul \
    --input="8x512x1024xf32=0.01" \
    --input="8x1024x512xf32=0.01" > ../../build/.output 2>&1

SyntaxError: invalid decimal literal (1338158115.py, line 2)

## Common Optimisations - shared with CPU as well
* Tile and distribute to workgroups
* Further tile the parallel dimensions
* TODO: give CPU and GPU pipelines as they are
* TODO: use two files for CPU and GPU opt.

### Tile and distribute to workgroups
* distribute to multi-processors (CPU) or workgroups (GPU) {Stream Multi-processors for CUDA specifically, since workgroups is the concept used by OpenCL and Vulkan}
* NOTE: only parallel dimensions will be tiled at this stage, to avoid sync between processors / workgroups
* HPs: workgroup-sizes, and processor-counts

In [ ]:
// -----// IR Dump After TileAndDistributeToWorkgroups (iree-codegen-tile-and-distribute-to-workgroups) //----- //
hal.executable.variant public @cuda_nvptx_fb, target = <"cuda", "cuda-nvptx-fb", {target_arch = "sm_70"}> {
  hal.executable.export public @matmul_dispatch_0_batch_matmul_8x512x512x1024_f32 ordinal(0) layout(#hal.pipeline.layout<push_constants = 0, sets = [<0, bindings = [<0, storage_buffer, ReadOnly>, <1, storage_buffer, ReadOnly>, <2, storage_buffer>]>]>) attributes {translation_info = #iree_codegen.translation_info<LLVMGPUMatmulSimt>, workgroup_size = [32 : index, 8 : index, 1 : index]} {
  ^bb0(%arg0: !hal.device):
    %c512 = arith.constant 512 : index
    %c1 = arith.constant 1 : index
    hal.return %c512, %c1, %c1 : index, index, index
  }
  builtin.module {
    func.func @matmul_dispatch_0_batch_matmul_8x512x512x1024_f32() {
      %c32 = arith.constant 32 : index
      %c128 = arith.constant 128 : index
      %c0 = arith.constant 0 : index
      %cst = arith.constant 0.000000e+00 : f32
      %0 = hal.interface.binding.subspan set(0) binding(0) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : !flow.dispatch.tensor<readonly:tensor<8x512x1024xf32>>
      %1 = hal.interface.binding.subspan set(0) binding(1) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : !flow.dispatch.tensor<readonly:tensor<8x1024x512xf32>>
      %2 = hal.interface.binding.subspan set(0) binding(2) type(storage_buffer) alignment(64) offset(%c0) : !flow.dispatch.tensor<writeonly:tensor<8x512x512xf32>>
      %workgroup_id_x = hal.interface.workgroup.id[0] : index
      %3 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
      %4 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x]
      %5 = flow.dispatch.tensor.load %0, offsets = [%3, %4, 0], sizes = [1, %c32, 1024], strides = [1, 1, 1] : !flow.dispatch.tensor<readonly:tensor<8x512x1024xf32>> -> tensor<1x?x1024xf32>
      %6 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
      %7 = affine.apply affine_map<()[s0] -> (s0 * 128 - (s0 floordiv 4) * 512)>()[%workgroup_id_x]
      %8 = flow.dispatch.tensor.load %1, offsets = [%6, 0, %7], sizes = [1, 1024, %c128], strides = [1, 1, 1] : !flow.dispatch.tensor<readonly:tensor<8x1024x512xf32>> -> tensor<1x1024x?xf32>
      %9 = tensor.empty() : tensor<1x32x128xf32>
      %10 = linalg.fill {lowering_config = #iree_codegen.lowering_config<tile_sizes = [[1, 32, 128, 32]]>} ins(%cst : f32) outs(%9 : tensor<1x32x128xf32>) -> tensor<1x32x128xf32>
      %cast = tensor.cast %8 : tensor<1x1024x?xf32> to tensor<1x1024x128xf32>
      %cast_0 = tensor.cast %5 : tensor<1x?x1024xf32> to tensor<1x32x1024xf32>
      %11 = linalg.batch_matmul {lowering_config = #iree_codegen.lowering_config<tile_sizes = [[1, 32, 128, 32]]>} ins(%cast_0, %cast : tensor<1x32x1024xf32>, tensor<1x1024x128xf32>) outs(%10 : tensor<1x32x128xf32>) -> tensor<1x32x128xf32>
      %cast_1 = tensor.cast %11 : tensor<1x32x128xf32> to tensor<1x?x?xf32>
      %12 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
      %13 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x]
      %14 = affine.apply affine_map<()[s0] -> (s0 * 128 - (s0 floordiv 4) * 512)>()[%workgroup_id_x]
      flow.dispatch.tensor.store %cast_1, %2, offsets = [%12, %13, %14], sizes = [1, %c32, %c128], strides = [1, 1, 1] : tensor<1x?x?xf32> -> !flow.dispatch.tensor<writeonly:tensor<8x512x512xf32>>
      return
    }
  }
}


### Further tile to parallel dimensions
This is the floating area for dataflow optimisations, they will introduce slicing operations, such as tensor.extract_slice / tensor.insert_slice in MLIR or chunkAt in CHOREO
* only to parallel dimensions
* HPs: selected tile sizes
* introduce: slicing operations, padding operations at L2 level

### Tile and distribute to workitems / Wraps (GPU only)
* tile on reduction dimensions (why?)
* Copy subviews of input memrefs to shared memory 
(workgroup memory) on the GPU prior to computatio; Insert barriers after copying to shared (workgroup) memory (32 MB to 128 MB, the remains are configured to L1 cache)
* Do an additional level of tiling to distribute to a warps (wrap size = 32 for Tesla)n

In [ ]:
// -----// IR Dump After GPUTensorTile (iree-codegen-gpu-tensor-tile) //----- //
func.func @matmul_dispatch_0_batch_matmul_8x512x512x1024_f32() {
  %c0 = arith.constant 0 : index
  %c32 = arith.constant 32 : index
  %c1024 = arith.constant 1024 : index
  %cst = arith.constant 0.000000e+00 : f32
  %0 = hal.interface.binding.subspan set(0) binding(0) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : !flow.dispatch.tensor<readonly:tensor<8x512x1024xf32>>
  %1 = hal.interface.binding.subspan set(0) binding(1) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : !flow.dispatch.tensor<readonly:tensor<8x1024x512xf32>>
  %2 = hal.interface.binding.subspan set(0) binding(2) type(storage_buffer) alignment(64) offset(%c0) : !flow.dispatch.tensor<writeonly:tensor<8x512x512xf32>>
  %workgroup_id_x = hal.interface.workgroup.id[0] : index
  %3 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
  %4 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x]
  %5 = affine.apply affine_map<()[s0] -> (s0 * 128 - (s0 floordiv 4) * 512)>()[%workgroup_id_x]
  %6 = flow.dispatch.tensor.load %2, offsets = [%3, %4, %5], sizes = [1, 32, 128], strides = [1, 1, 1] : !flow.dispatch.tensor<writeonly:tensor<8x512x512xf32>> -> tensor<1x32x128xf32>
  %7 = flow.dispatch.tensor.load %0, offsets = [%3, %4, 0], sizes = [1, 32, 1024], strides = [1, 1, 1] : !flow.dispatch.tensor<readonly:tensor<8x512x1024xf32>> -> tensor<1x32x1024xf32>
  %8 = flow.dispatch.tensor.load %1, offsets = [%3, 0, %5], sizes = [1, 1024, 128], strides = [1, 1, 1] : !flow.dispatch.tensor<readonly:tensor<8x1024x512xf32>> -> tensor<1x1024x128xf32>
  %9 = scf.forall (%arg0, %arg1) in (8, 32) shared_outs(%arg2 = %6) -> (tensor<1x32x128xf32>) {
    %11 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg0)
    %12 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg1)
    %extracted_slice = tensor.extract_slice %arg2[0, %11, %12] [1, 4, 4] [1, 1, 1] : tensor<1x32x128xf32> to tensor<1x4x4xf32>
    %13 = linalg.fill {__internal_linalg_transform__ = "workgroup_k_tiled", lowering_config = #iree_codegen.lowering_config<tile_sizes = [[1, 32, 128, 32]]>} ins(%cst : f32) outs(%extracted_slice : tensor<1x4x4xf32>) -> tensor<1x4x4xf32>
    %14 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg0)
    %15 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg1)
    scf.forall.in_parallel {
      tensor.parallel_insert_slice %13 into %arg2[0, %14, %15] [1, 4, 4] [1, 1, 1] : tensor<1x4x4xf32> into tensor<1x32x128xf32>
    }
  } {mapping = [#gpu.thread<y>, #gpu.thread<x>]}
  %10 = scf.for %arg0 = %c0 to %c1024 step %c32 iter_args(%arg1 = %9) -> (tensor<1x32x128xf32>) {
    %extracted_slice = tensor.extract_slice %7[0, 0, %arg0] [1, 32, 32] [1, 1, 1] : tensor<1x32x1024xf32> to tensor<1x32x32xf32>
    %extracted_slice_0 = tensor.extract_slice %8[0, %arg0, 0] [1, 32, 128] [1, 1, 1] : tensor<1x1024x128xf32> to tensor<1x32x128xf32>
    %11 = bufferization.alloc_tensor() copy(%extracted_slice) {bufferization.escape = [false]} : tensor<1x32x32xf32>
    %12 = bufferization.alloc_tensor() copy(%extracted_slice_0) {bufferization.escape = [false]} : tensor<1x32x128xf32>
    %13 = scf.forall (%arg2, %arg3) in (8, 32) shared_outs(%arg4 = %arg1) -> (tensor<1x32x128xf32>) {
      %14 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg2)
      %15 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg3)
      %16 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg2)
      %17 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg3)
      %extracted_slice_1 = tensor.extract_slice %11[0, %14, 0] [1, 4, 32] [1, 1, 1] : tensor<1x32x32xf32> to tensor<1x4x32xf32>
      %extracted_slice_2 = tensor.extract_slice %12[0, 0, %15] [1, 32, 4] [1, 1, 1] : tensor<1x32x128xf32> to tensor<1x32x4xf32>
      %extracted_slice_3 = tensor.extract_slice %arg4[0, %16, %17] [1, 4, 4] [1, 1, 1] : tensor<1x32x128xf32> to tensor<1x4x4xf32>
      %18 = linalg.batch_matmul {__internal_linalg_transform__ = "workgroup_k_tiled", lowering_config = #iree_codegen.lowering_config<tile_sizes = [[1, 32, 128, 32]]>} ins(%extracted_slice_1, %extracted_slice_2 : tensor<1x4x32xf32>, tensor<1x32x4xf32>) outs(%extracted_slice_3 : tensor<1x4x4xf32>) -> tensor<1x4x4xf32>
      %19 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg2)
      %20 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg3)
      scf.forall.in_parallel {
        tensor.parallel_insert_slice %18 into %arg4[0, %19, %20] [1, 4, 4] [1, 1, 1] : tensor<1x4x4xf32> into tensor<1x32x128xf32>
      }
    } {mapping = [#gpu.thread<y>, #gpu.thread<x>]}
    scf.yield %13 : tensor<1x32x128xf32>
  }
  flow.dispatch.tensor.store %10, %2, offsets = [%3, %4, %5], sizes = [1, 32, 128], strides = [1, 1, 1] : tensor<1x32x128xf32> -> !flow.dispatch.tensor<writeonly:tensor<8x512x512xf32>>
  return
}


### Vectorize
* emits vector.transfer_read and transfer_write operations for load/store ops
* fix some tensor_slice (must be contiguous) by lowering to vector type
* use vector calc operations, such as vector.vmulf, vector.contract
* use vector.broadcast and vector.transpose to bridging between data patterns, they are more close to the HW dma abstraction then high-dimensional tensor permutations

In [ ]:
// -----// IR Dump After GPUVectorization (iree-codegen-gpu-vectorization) //----- //
func.func @matmul_dispatch_0_batch_matmul_8x512x512x1024_f32() {
  %cst = arith.constant dense<0.000000e+00> : vector<1x4x4xf32>
  %c0 = arith.constant 0 : index
  %c32 = arith.constant 32 : index
  %c1024 = arith.constant 1024 : index
  %cst_0 = arith.constant 0.000000e+00 : f32
  %0 = hal.interface.binding.subspan set(0) binding(0) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : !flow.dispatch.tensor<readonly:tensor<8x512x1024xf32>>
  %1 = hal.interface.binding.subspan set(0) binding(1) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : !flow.dispatch.tensor<readonly:tensor<8x1024x512xf32>>
  %2 = hal.interface.binding.subspan set(0) binding(2) type(storage_buffer) alignment(64) offset(%c0) : !flow.dispatch.tensor<writeonly:tensor<8x512x512xf32>>
  %workgroup_id_x = hal.interface.workgroup.id[0] : index
  %3 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
  %4 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x]
  %5 = affine.apply affine_map<()[s0] -> (s0 * 128 - (s0 floordiv 4) * 512)>()[%workgroup_id_x]
  %6 = flow.dispatch.tensor.load %2, offsets = [%3, %4, %5], sizes = [1, 32, 128], strides = [1, 1, 1] : !flow.dispatch.tensor<writeonly:tensor<8x512x512xf32>> -> tensor<1x32x128xf32>
  %7 = flow.dispatch.tensor.load %0, offsets = [%3, %4, 0], sizes = [1, 32, 1024], strides = [1, 1, 1] : !flow.dispatch.tensor<readonly:tensor<8x512x1024xf32>> -> tensor<1x32x1024xf32>
  %8 = flow.dispatch.tensor.load %1, offsets = [%3, 0, %5], sizes = [1, 1024, 128], strides = [1, 1, 1] : !flow.dispatch.tensor<readonly:tensor<8x1024x512xf32>> -> tensor<1x1024x128xf32>
  %9 = scf.forall (%arg0, %arg1) in (8, 32) shared_outs(%arg2 = %6) -> (tensor<1x32x128xf32>) {
    %11 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg0)
    %12 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg1)
    %extracted_slice = tensor.extract_slice %arg2[0, %11, %12] [1, 4, 4] [1, 1, 1] : tensor<1x32x128xf32> to tensor<1x4x4xf32>
    %13 = vector.transfer_write %cst, %extracted_slice[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, tensor<1x4x4xf32>
    %14 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg0)
    %15 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg1)
    scf.forall.in_parallel {
      tensor.parallel_insert_slice %13 into %arg2[0, %14, %15] [1, 4, 4] [1, 1, 1] : tensor<1x4x4xf32> into tensor<1x32x128xf32>
    }
  } {mapping = [#gpu.thread<y>, #gpu.thread<x>]}
  %10 = scf.for %arg0 = %c0 to %c1024 step %c32 iter_args(%arg1 = %9) -> (tensor<1x32x128xf32>) {
    %extracted_slice = tensor.extract_slice %7[0, 0, %arg0] [1, 32, 32] [1, 1, 1] : tensor<1x32x1024xf32> to tensor<1x32x32xf32>
    %extracted_slice_1 = tensor.extract_slice %8[0, %arg0, 0] [1, 32, 128] [1, 1, 1] : tensor<1x1024x128xf32> to tensor<1x32x128xf32>
    %11 = bufferization.alloc_tensor() copy(%extracted_slice) {bufferization.escape = [false]} : tensor<1x32x32xf32>
    %12 = bufferization.alloc_tensor() copy(%extracted_slice_1) {bufferization.escape = [false]} : tensor<1x32x128xf32>
    %13 = scf.forall (%arg2, %arg3) in (8, 32) shared_outs(%arg4 = %arg1) -> (tensor<1x32x128xf32>) {
      %14 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg2)
      %15 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg3)
      %16 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg2)
      %17 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg3)
      %extracted_slice_2 = tensor.extract_slice %11[0, %14, 0] [1, 4, 32] [1, 1, 1] : tensor<1x32x32xf32> to tensor<1x4x32xf32>
      %extracted_slice_3 = tensor.extract_slice %12[0, 0, %15] [1, 32, 4] [1, 1, 1] : tensor<1x32x128xf32> to tensor<1x32x4xf32>
      %extracted_slice_4 = tensor.extract_slice %arg4[0, %16, %17] [1, 4, 4] [1, 1, 1] : tensor<1x32x128xf32> to tensor<1x4x4xf32>
      %18 = vector.transfer_read %extracted_slice_2[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : tensor<1x4x32xf32>, vector<1x4x32xf32>
      %19 = vector.transfer_read %extracted_slice_3[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : tensor<1x32x4xf32>, vector<1x32x4xf32>
      %20 = vector.transfer_read %extracted_slice_4[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : tensor<1x4x4xf32>, vector<1x4x4xf32>
      %21 = vector.contract {indexing_maps = [affine_map<(d0, d1, d2, d3) -> (d0, d1, d3)>, affine_map<(d0, d1, d2, d3) -> (d0, d3, d2)>, affine_map<(d0, d1, d2, d3) -> (d0, d1, d2)>], iterator_types = ["parallel", "parallel", "parallel", "reduction"], kind = #vector.kind<add>} %18, %19, %20 : vector<1x4x32xf32>, vector<1x32x4xf32> into vector<1x4x4xf32>
      %22 = vector.transfer_write %21, %extracted_slice_4[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, tensor<1x4x4xf32>
      %23 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg2)
      %24 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg3)
      scf.forall.in_parallel {
        tensor.parallel_insert_slice %22 into %arg4[0, %23, %24] [1, 4, 4] [1, 1, 1] : tensor<1x4x4xf32> into tensor<1x32x128xf32>
      }
    } {mapping = [#gpu.thread<y>, #gpu.thread<x>]}
    scf.yield %13 : tensor<1x32x128xf32>
  }
  flow.dispatch.tensor.store %10, %2, offsets = [%3, %4, %5], sizes = [1, 32, 128], strides = [1, 1, 1] : tensor<1x32x128xf32> -> !flow.dispatch.tensor<writeonly:tensor<8x512x512xf32>>
  return
}


### Do SM copy vectorisation (GPU only)
* use vector.transfer_read and transfer_write
* will map to GPU's vector load and store on shared memory
* vector length is set according to GPU, like 128 bits for coalesced access
* unroll vector load/stores by the targeting load/store size prior step

In [ ]:
// -----// IR Dump After GPUDistribute (iree-codegen-gpu-distribute) //----- //
func.func @matmul_dispatch_0_batch_matmul_8x512x512x1024_f32() {
  %c0 = arith.constant 0 : index
  %cst = arith.constant dense<0.000000e+00> : vector<1x4x4xf32>
  %c0_0 = arith.constant 0 : index
  %c32 = arith.constant 32 : index
  %c1024 = arith.constant 1024 : index
  %cst_1 = arith.constant 0.000000e+00 : f32
  %0 = hal.interface.binding.subspan set(0) binding(0) type(storage_buffer) alignment(64) offset(%c0_0) flags(ReadOnly) : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %0, 64 : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  %1 = hal.interface.binding.subspan set(0) binding(1) type(storage_buffer) alignment(64) offset(%c0_0) flags(ReadOnly) : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %1, 64 : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  %2 = hal.interface.binding.subspan set(0) binding(2) type(storage_buffer) alignment(64) offset(%c0_0) : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %2, 64 : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  %workgroup_id_x = hal.interface.workgroup.id[0] : index
  %3 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
  %4 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x]
  %5 = affine.apply affine_map<()[s0] -> (s0 * 128 - (s0 floordiv 4) * 512)>()[%workgroup_id_x]
  %subview = memref.subview %2[%3, %4, %5] [1, 32, 128] [1, 1, 1] : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  %subview_2 = memref.subview %0[%3, %4, 0] [1, 32, 1024] [1, 1, 1] : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>> to memref<1x32x1024xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  %subview_3 = memref.subview %1[%3, 0, %5] [1, 1024, 128] [1, 1, 1] : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>> to memref<1x1024x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  %6 = gpu.thread_id  x
  %7 = gpu.thread_id  y
  %8 = gpu.thread_id  z
  %9 = affine.apply affine_map<(d0) -> (d0 * 4)>(%7)
  %10 = affine.apply affine_map<(d0) -> (d0 * 4)>(%6)
  %subview_4 = memref.subview %subview[0, %9, %10] [1, 4, 4] [1, 1, 1] : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  vector.transfer_write %cst, %subview_4[%c0_0, %c0_0, %c0_0] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  scf.for %arg0 = %c0_0 to %c1024 step %c32 {
    %subview_5 = memref.subview %subview_2[0, 0, %arg0] [1, 32, 32] [1, 1, 1] : memref<1x32x1024xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x32xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_6 = memref.subview %subview_3[0, %arg0, 0] [1, 32, 128] [1, 1, 1] : memref<1x1024x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %alloc = memref.alloc() : memref<1x32x32xf32, #gpu.address_space<workgroup>>
    gpu.barrier
    memref.copy %subview_5, %alloc {__internal_linalg_transform__ = "copy_to_workgroup_memory"} : memref<1x32x32xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x32xf32, #gpu.address_space<workgroup>>
    gpu.barrier
    %alloc_7 = memref.alloc() : memref<1x32x128xf32, #gpu.address_space<workgroup>>
    gpu.barrier
    memref.copy %subview_6, %alloc_7 {__internal_linalg_transform__ = "copy_to_workgroup_memory"} : memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, #gpu.address_space<workgroup>>
    gpu.barrier
    %11 = gpu.thread_id  x
    %12 = gpu.thread_id  y
    %13 = gpu.thread_id  z
    %14 = affine.apply affine_map<(d0) -> (d0 * 4)>(%12)
    %15 = affine.apply affine_map<(d0) -> (d0 * 4)>(%11)
    %subview_8 = memref.subview %alloc[0, %14, 0] [1, 4, 32] [1, 1, 1] : memref<1x32x32xf32, #gpu.address_space<workgroup>> to memref<1x4x32xf32, strided<[1024, 32, 1], offset: ?>, #gpu.address_space<workgroup>>
    %subview_9 = memref.subview %alloc_7[0, 0, %15] [1, 32, 4] [1, 1, 1] : memref<1x32x128xf32, #gpu.address_space<workgroup>> to memref<1x32x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
    %subview_10 = memref.subview %subview[0, %14, %15] [1, 4, 4] [1, 1, 1] : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %16 = vector.transfer_read %subview_8[%c0_0, %c0_0, %c0_0], %cst_1 {in_bounds = [true, true, true]} : memref<1x4x32xf32, strided<[1024, 32, 1], offset: ?>, #gpu.address_space<workgroup>>, vector<1x4x32xf32>
    %17 = vector.transfer_read %subview_9[%c0_0, %c0_0, %c0_0], %cst_1 {in_bounds = [true, true, true]} : memref<1x32x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>, vector<1x32x4xf32>
    %18 = vector.transfer_read %subview_10[%c0_0, %c0_0, %c0_0], %cst_1 {in_bounds = [true, true, true]} : memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x4x4xf32>
    %19 = vector.contract {indexing_maps = [affine_map<(d0, d1, d2, d3) -> (d0, d1, d3)>, affine_map<(d0, d1, d2, d3) -> (d0, d3, d2)>, affine_map<(d0, d1, d2, d3) -> (d0, d1, d2)>], iterator_types = ["parallel", "parallel", "parallel", "reduction"], kind = #vector.kind<add>} %16, %17, %18 : vector<1x4x32xf32>, vector<1x32x4xf32> into vector<1x4x4xf32>
    vector.transfer_write %19, %subview_10[%c0_0, %c0_0, %c0_0] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  }
  return
}
// -----// IR Dump After MemrefCopyToLinalgPass (iree-codegen-memrefcopy-to-linalg) //----- //
func.func @matmul_dispatch_0_batch_matmul_8x512x512x1024_f32() {
  %c0 = arith.constant 0 : index
  %cst = arith.constant dense<0.000000e+00> : vector<1x4x4xf32>
  %c32 = arith.constant 32 : index
  %c1024 = arith.constant 1024 : index
  %cst_0 = arith.constant 0.000000e+00 : f32
  %0 = hal.interface.binding.subspan set(0) binding(0) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %0, 64 : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  %1 = hal.interface.binding.subspan set(0) binding(1) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %1, 64 : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  %2 = hal.interface.binding.subspan set(0) binding(2) type(storage_buffer) alignment(64) offset(%c0) : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %2, 64 : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  %workgroup_id_x = hal.interface.workgroup.id[0] : index
  %3 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
  %4 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x]
  %5 = affine.apply affine_map<()[s0] -> (s0 * 128 - (s0 floordiv 4) * 512)>()[%workgroup_id_x]
  %subview = memref.subview %2[%3, %4, %5] [1, 32, 128] [1, 1, 1] : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  %subview_1 = memref.subview %0[%3, %4, 0] [1, 32, 1024] [1, 1, 1] : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>> to memref<1x32x1024xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  %subview_2 = memref.subview %1[%3, 0, %5] [1, 1024, 128] [1, 1, 1] : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>> to memref<1x1024x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  %6 = gpu.thread_id  x
  %7 = gpu.thread_id  y
  %8 = affine.apply affine_map<(d0) -> (d0 * 4)>(%7)
  %9 = affine.apply affine_map<(d0) -> (d0 * 4)>(%6)
  %subview_3 = memref.subview %subview[0, %8, %9] [1, 4, 4] [1, 1, 1] : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  vector.transfer_write %cst, %subview_3[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  scf.for %arg0 = %c0 to %c1024 step %c32 {
    %subview_4 = memref.subview %subview_1[0, 0, %arg0] [1, 32, 32] [1, 1, 1] : memref<1x32x1024xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x32xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_5 = memref.subview %subview_2[0, %arg0, 0] [1, 32, 128] [1, 1, 1] : memref<1x1024x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %alloc = memref.alloc() : memref<1x32x32xf32, #gpu.address_space<workgroup>>
    gpu.barrier
    linalg.generic {indexing_maps = [affine_map<(d0, d1, d2) -> (d0, d1, d2)>, affine_map<(d0, d1, d2) -> (d0, d1, d2)>], iterator_types = ["parallel", "parallel", "parallel"]} ins(%subview_4 : memref<1x32x32xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>) outs(%alloc : memref<1x32x32xf32, #gpu.address_space<workgroup>>) attrs =  {__internal_linalg_transform__ = "copy_to_workgroup_memory"} {
    ^bb0(%in: f32, %out: f32):
      linalg.yield %in : f32
    }
    gpu.barrier
    %alloc_6 = memref.alloc() : memref<1x32x128xf32, #gpu.address_space<workgroup>>
    gpu.barrier
    linalg.generic {indexing_maps = [affine_map<(d0, d1, d2) -> (d0, d1, d2)>, affine_map<(d0, d1, d2) -> (d0, d1, d2)>], iterator_types = ["parallel", "parallel", "parallel"]} ins(%subview_5 : memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>) outs(%alloc_6 : memref<1x32x128xf32, #gpu.address_space<workgroup>>) attrs =  {__internal_linalg_transform__ = "copy_to_workgroup_memory"} {
    ^bb0(%in: f32, %out: f32):
      linalg.yield %in : f32
    }
    gpu.barrier
    %10 = gpu.thread_id  x
    %11 = gpu.thread_id  y
    %12 = affine.apply affine_map<(d0) -> (d0 * 4)>(%11)
    %13 = affine.apply affine_map<(d0) -> (d0 * 4)>(%10)
    %subview_7 = memref.subview %alloc[0, %12, 0] [1, 4, 32] [1, 1, 1] : memref<1x32x32xf32, #gpu.address_space<workgroup>> to memref<1x4x32xf32, strided<[1024, 32, 1], offset: ?>, #gpu.address_space<workgroup>>
    %subview_8 = memref.subview %alloc_6[0, 0, %13] [1, 32, 4] [1, 1, 1] : memref<1x32x128xf32, #gpu.address_space<workgroup>> to memref<1x32x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
    %subview_9 = memref.subview %subview[0, %12, %13] [1, 4, 4] [1, 1, 1] : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %14 = vector.transfer_read %subview_7[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x4x32xf32, strided<[1024, 32, 1], offset: ?>, #gpu.address_space<workgroup>>, vector<1x4x32xf32>
    %15 = vector.transfer_read %subview_8[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x32x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>, vector<1x32x4xf32>
    %16 = vector.transfer_read %subview_9[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x4x4xf32>
    %17 = vector.contract {indexing_maps = [affine_map<(d0, d1, d2, d3) -> (d0, d1, d3)>, affine_map<(d0, d1, d2, d3) -> (d0, d3, d2)>, affine_map<(d0, d1, d2, d3) -> (d0, d1, d2)>], iterator_types = ["parallel", "parallel", "parallel", "reduction"], kind = #vector.kind<add>} %14, %15, %16 : vector<1x4x32xf32>, vector<1x32x4xf32> into vector<1x4x4xf32>
, kind = #vector.kind<add>} %14, %15, %16 : vector<1x4x32xf32>, vector<1x32x4xf32> into vector<1x4x4xf32>
    vector.transfer_write %17, %subview_9[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  }
  return
}

// -----// IR Dump After GPUDistributeSharedMemoryCopy (iree-codegen-gpu-distribute-shared-memory-copy) //----- //
func.func @matmul_dispatch_0_batch_matmul_8x512x512x1024_f32() {
  %c0 = arith.constant 0 : index
  %cst = arith.constant dense<0.000000e+00> : vector<1x4x4xf32>
  %c32 = arith.constant 32 : index
  %c1024 = arith.constant 1024 : index
  %cst_0 = arith.constant 0.000000e+00 : f32
  %c128 = arith.constant 128 : index
  %c8 = arith.constant 8 : index
  %c1 = arith.constant 1 : index
  %0 = gpu.thread_id  x
  %1 = gpu.thread_id  y
  %2 = gpu.thread_id  z
  %alloc = memref.alloc() : memref<1x32x128xf32, #gpu.address_space<workgroup>>
  %alloc_1 = memref.alloc() : memref<1x32x32xf32, #gpu.address_space<workgroup>>
  %3 = hal.interface.binding.subspan set(0) binding(0) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %3, 64 : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  %4 = hal.interface.binding.subspan set(0) binding(1) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %4, 64 : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  %5 = hal.interface.binding.subspan set(0) binding(2) type(storage_buffer) alignment(64) offset(%c0) : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %5, 64 : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  %workgroup_id_x = hal.interface.workgroup.id[0] : index
  %6 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
  %7 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x]
  %8 = affine.apply affine_map<()[s0] -> (s0 * 128 - (s0 floordiv 4) * 512)>()[%workgroup_id_x]
  %subview = memref.subview %5[%6, %7, %8] [1, 32, 128] [1, 1, 1] : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  %subview_2 = memref.subview %3[%6, %7, 0] [1, 32, 1024] [1, 1, 1] : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>> to memref<1x32x1024xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  %subview_3 = memref.subview %4[%6, 0, %8] [1, 1024, 128] [1, 1, 1] : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>> to memref<1x1024x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  %9 = gpu.thread_id  x
  %10 = gpu.thread_id  y
  %11 = affine.apply affine_map<(d0) -> (d0 * 4)>(%10)
  %12 = affine.apply affine_map<(d0) -> (d0 * 4)>(%9)
  %subview_4 = memref.subview %subview[0, %11, %12] [1, 4, 4] [1, 1, 1] : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  vector.transfer_write %cst, %subview_4[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  scf.for %arg0 = %c0 to %c1024 step %c32 {
    %subview_5 = memref.subview %subview_2[0, 0, %arg0] [1, 32, 32] [1, 1, 1] : memref<1x32x1024xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x32xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_6 = memref.subview %subview_3[0, %arg0, 0] [1, 32, 128] [1, 1, 1] : memref<1x1024x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    gpu.barrier
    %subview_7 = memref.subview %alloc_1[%c0, %c0, %c0] [1, 32, 32] [1, 1, 1] : memref<1x32x32xf32, #gpu.address_space<workgroup>> to memref<1x32x32xf32, strided<[1024, 32, 1], offset: ?>, #gpu.address_space<workgroup>>
    %13 = affine.apply affine_map<()[s0, s1, s2] -> (s1 * 4 + s2 * 32 + s0 floordiv 8)>()[%0, %1, %2]
    %14 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 8) * 32)>()[%0]
    %15 = affine.apply affine_map<()[s0, s1, s2] -> (s1 * 4 + s2 * 32 + s0 floordiv 8)>()[%0, %1, %2]
    %16 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 8) * 32)>()[%0]
    %subview_8 = memref.subview %subview_5[0, %13, %14] [1, 1, 4] [1, 1, 1] : memref<1x32x32xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x1x4xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_9 = memref.subview %subview_7[0, %15, %16] [1, 1, 4] [1, 1, 1] : memref<1x32x32xf32, strided<[1024, 32, 1], offset: ?>, #gpu.address_space<workgroup>> to memref<1x1x4xf32, strided<[1024, 32, 1], offset: ?>, #gpu.address_space<workgroup>>
    %17 = vector.transfer_read %subview_8[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x1x4xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    vector.transfer_write %17, %subview_9[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x1x4xf32, strided<[1024, 32, 1], offset: ?>, #gpu.address_space<workgroup>>
    %c32_10 = arith.constant 32 : index
    %subview_11 = memref.subview %subview_6[%c0, %c0, %c0] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_12 = memref.subview %alloc[%c0, %c0, %c0] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, #gpu.address_space<workgroup>> to memref<1x8x128xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
    %18 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32)>()[%0, %1, %2]
    %19 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
    %20 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32)>()[%0, %1, %2]
    %21 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
    %subview_13 = memref.subview %subview_11[0, %18, %19] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_14 = memref.subview %subview_12[0, %20, %21] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>> to memref<1x1x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
    %22 = vector.transfer_read %subview_13[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    vector.transfer_write %22, %subview_14[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x1x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
    %c1_15 = arith.constant 1 : index
    %23 = arith.muli %c8, %c1_15 : index
    %24 = arith.addi %c0, %23 : index
    scf.for %arg1 = %c0 to %c128 step %c128 {
      %subview_19 = memref.subview %subview_6[%c0, %24, %arg1] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
      %subview_20 = memref.subview %alloc[%c0, %24, %arg1] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, #gpu.address_space<workgroup>> to memref<1x8x128xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
      %37 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32)>()[%0, %1, %2]
      %38 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
      %39 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32)>()[%0, %1, %2]
      %40 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
      %subview_21 = memref.subview %subview_19[0, %37, %38] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
      %subview_22 = memref.subview %subview_20[0, %39, %40] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>> to memref<1x1x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
      %41 = vector.transfer_read %subview_21[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
      vector.transfer_write %41, %subview_22[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x1x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
    }
    %c2 = arith.constant 2 : index
    %25 = arith.muli %c8, %c2 : index
    %26 = arith.addi %c0, %25 : index
    scf.for %arg1 = %c0 to %c128 step %c128 {
      %subview_19 = memref.subview %subview_6[%c0, %26, %arg1] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
      %subview_20 = memref.subview %alloc[%c0, %26, %arg1] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, #gpu.address_space<workgroup>> to memref<1x8x128xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
      %37 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32)>()[%0, %1, %2]
      %38 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
      %39 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32)>()[%0, %1, %2]
      %40 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
      %subview_21 = memref.subview %subview_19[0, %37, %38] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
      %subview_22 = memref.subview %subview_20[0, %39, %40] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>> to memref<1x1x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
      %41 = vector.transfer_read %subview_21[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
      vector.transfer_write %41, %subview_22[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x1x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
    }
    %c3 = arith.constant 3 : index
    %27 = arith.muli %c8, %c3 : index
    %28 = arith.addi %c0, %27 : index
    scf.for %arg1 = %c0 to %c128 step %c128 {
      %subview_19 = memref.subview %subview_6[%c0, %28, %arg1] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
      %subview_20 = memref.subview %alloc[%c0, %28, %arg1] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, #gpu.address_space<workgroup>> to memref<1x8x128xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
      %37 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32)>()[%0, %1, %2]
      %38 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
      %39 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32)>()[%0, %1, %2]
      %40 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
      %subview_21 = memref.subview %subview_19[0, %37, %38] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
tor_type<storage_buffer>>
      %subview_22 = memref.subview %subview_20[0, %39, %40] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>> to memref<1x1x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
      %41 = vector.transfer_read %subview_21[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
      vector.transfer_write %41, %subview_22[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x1x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
    }
    gpu.barrier
    %29 = gpu.thread_id  x
    %30 = gpu.thread_id  y
    %31 = affine.apply affine_map<(d0) -> (d0 * 4)>(%30)
    %32 = affine.apply affine_map<(d0) -> (d0 * 4)>(%29)
    %subview_16 = memref.subview %alloc_1[0, %31, 0] [1, 4, 32] [1, 1, 1] : memref<1x32x32xf32, #gpu.address_space<workgroup>> to memref<1x4x32xf32, strided<[1024, 32, 1], offset: ?>, #gpu.address_space<workgroup>>
    %subview_17 = memref.subview %alloc[0, 0, %32] [1, 32, 4] [1, 1, 1] : memref<1x32x128xf32, #gpu.address_space<workgroup>> to memref<1x32x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
    %subview_18 = memref.subview %subview[0, %31, %32] [1, 4, 4] [1, 1, 1] : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %33 = vector.transfer_read %subview_16[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x4x32xf32, strided<[1024, 32, 1], offset: ?>, #gpu.address_space<workgroup>>, vector<1x4x32xf32>
    %34 = vector.transfer_read %subview_17[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x32x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>, vector<1x32x4xf32>
    %35 = vector.transfer_read %subview_18[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x4x4xf32>
    %36 = vector.contract {indexing_maps = [affine_map<(d0, d1, d2, d3) -> (d0, d1, d3)>, affine_map<(d0, d1, d2, d3) -> (d0, d3, d2)>, affine_map<(d0, d1, d2, d3) -> (d0, d1, d2)>], iterator_types = ["parallel", "parallel", "parallel", "reduction"], kind = #vector.kind<add>} %33, %34, %35 : vector<1x4x32xf32>, vector<1x32x4xf32> into vector<1x4x4xf32>
    vector.transfer_write %36, %subview_18[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  }
  return
}


### Multi-Buffering (or Double buffering) (GPU only)
* In order to hide latency, we can use double/multi-buffering to break dependencies between consecutive iterations 
of a loop using the same temporary buffe
* prior step required for pipelining
* Number of copies determined by pipeline-depth
r

### Bufferisation for better memory access patterns
We then optimise the memory access patterns following these principles:
* Avoid alloc and copy, make them as little as possible, obvious as hell.
* explicitly reusing buffers with in-place style: this can be guided by destination passing style (which has longest lifetime during the kernel scope):
    * In-place bufferization is an optimization technique aimed at reducing the overhead of memory allocations and data copying. With in-place bufferization, operations are performed directly on the input data's original buffer when possible, rather than allocating a new buffer for the output results. This approach can lead to more efficient use of memory and reduced execution time of programs.
    * Destination-passing style is a programming paradigm where the location (the "destination") for the output of a function or operation is passed as an argument to the function. This contrasts with traditional approaches where a function returns a new data structure or object as its result. By using destination-passing style, functions can operate directly on the provided destination for their output, potentially avoiding unnecessary data copying and improving efficiency.
    * it suggests adopting destination-passing style as a method or heuristic rule to facilitate or implement in-place bufferization in compiler optimization and kernel programming. Specifically, compilers or programmers can design algorithms and functions to accept output buffers as parameters and then compute and store data directly in these buffers, thereby avoiding additional memory allocations and data copying.
* Tie output tensor to results tensor to act as 
bufferization constrait
    * By "tying" the output tensor to the results tensor, the idea is to use the same memory buffer for both the intermediate output and the final result of a computation. This acts as a constraint in the bufferization process, dictating that certain operations should reuse memory to minimize memory allocations and data copying.
    * This technique facilitates in-place operations, where computations modify data directly in the memory buffer of the input tensor instead of creating a new tensor for every intermediate step. In-place operations can significantly reduce the memory footprint of running deep learning models.
* Chaining consecutive ops, operands tied to precedessor's results, but do care about (RAW conflicts)n

In [ ]:
// -----// IR Dump After IREEComprehensiveBufferize (iree-codegen-iree-comprehensive-bufferize) //----- //
module {
  func.func @matmul_dispatch_0_batch_matmul_8x512x512x1024_f32() {
    %cst = arith.constant dense<0.000000e+00> : vector<1x4x4xf32>
    %c0 = arith.constant 0 : index
    %c32 = arith.constant 32 : index
    %c1024 = arith.constant 1024 : index
    %cst_0 = arith.constant 0.000000e+00 : f32
    %0 = hal.interface.binding.subspan set(0) binding(0) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
    memref.assume_alignment %0, 64 : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
    %1 = hal.interface.binding.subspan set(0) binding(1) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
    memref.assume_alignment %1, 64 : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
    %2 = hal.interface.binding.subspan set(0) binding(2) type(storage_buffer) alignment(64) offset(%c0) : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
    memref.assume_alignment %2, 64 : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
    %workgroup_id_x = hal.interface.workgroup.id[0] : index
    %3 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
    %4 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x]
    %5 = affine.apply affine_map<()[s0] -> (s0 * 128 - (s0 floordiv 4) * 512)>()[%workgroup_id_x]
    %subview = memref.subview %2[%3, %4, %5] [1, 32, 128] [1, 1, 1] : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_1 = memref.subview %0[%3, %4, 0] [1, 32, 1024] [1, 1, 1] : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>> to memref<1x32x1024xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_2 = memref.subview %1[%3, 0, %5] [1, 1024, 128] [1, 1, 1] : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>> to memref<1x1024x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    scf.forall (%arg0, %arg1) in (8, 32) {
      %7 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg0)
      %8 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg1)
      %subview_4 = memref.subview %subview[0, %7, %8] [1, 4, 4] [1, 1, 1] : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
      vector.transfer_write %cst, %subview_4[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
      %subview_5 = memref.subview %subview[0, %7, %8] [1, 4, 4] [1, 1, 1] : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
      memref.copy %subview_4, %subview_5 : memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    } {mapping = [#gpu.thread<y>, #gpu.thread<x>]}
    %6 = scf.for %arg0 = %c0 to %c1024 step %c32 iter_args(%arg1 = %subview) -> (memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>) {
      %subview_4 = memref.subview %subview_1[0, 0, %arg0] [1, 32, 32] [1, 1, 1] : memref<1x32x1024xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x32xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
      %subview_5 = memref.subview %subview_2[0, %arg0, 0] [1, 32, 128] [1, 1, 1] : memref<1x1024x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
      %alloc = memref.alloc() : memref<1x32x32xf32, #gpu.address_space<workgroup>>
      gpu.barrier
      memref.copy %subview_4, %alloc {__internal_linalg_transform__ = "copy_to_workgroup_memory"} : memref<1x32x32xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x32xf32, #gpu.address_space<workgroup>>
      gpu.barrier
      %alloc_6 = memref.alloc() : memref<1x32x128xf32, #gpu.address_space<workgroup>>
      gpu.barrier
      memref.copy %subview_5, %alloc_6 {__internal_linalg_transform__ = "copy_to_workgroup_memory"} : memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, #gpu.address_space<workgroup>>
      gpu.barrier
      scf.forall (%arg2, %arg3) in (8, 32) {
        %7 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg2)
        %8 = affine.apply affine_map<(d0) -> (d0 * 4)>(%arg3)
        %subview_7 = memref.subview %alloc[0, %7, 0] [1, 4, 32] [1, 1, 1] : memref<1x32x32xf32, #gpu.address_space<workgroup>> to memref<1x4x32xf32, strided<[1024, 32, 1], offset: ?>, #gpu.address_space<workgroup>>
        %subview_8 = memref.subview %alloc_6[0, 0, %8] [1, 32, 4] [1, 1, 1] : memref<1x32x128xf32, #gpu.address_space<workgroup>> to memref<1x32x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>
        %subview_9 = memref.subview %arg1[0, %7, %8] [1, 4, 4] [1, 1, 1] : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
        %9 = vector.transfer_read %subview_7[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x4x32xf32, strided<[1024, 32, 1], offset: ?>, #gpu.address_space<workgroup>>, vector<1x4x32xf32>
        %10 = vector.transfer_read %subview_8[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x32x4xf32, strided<[4096, 128, 1], offset: ?>, #gpu.address_space<workgroup>>, vector<1x32x4xf32>
        %11 = vector.transfer_read %subview_9[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x4x4xf32>
        %12 = vector.contract {indexing_maps = [affine_map<(d0, d1, d2, d3) -> (d0, d1, d3)>, affine_map<(d0, d1, d2, d3) -> (d0, d3, d2)>, affine_map<(d0, d1, d2, d3) -> (d0, d1, d2)>], iterator_types = ["parallel", "parallel", "parallel", "reduction"], kind = #vector.kind<add>} %9, %10, %11 : vector<1x4x32xf32>, vector<1x32x4xf32> into vector<1x4x4xf32>
        vector.transfer_write %12, %subview_9[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
        %subview_10 = memref.subview %arg1[0, %7, %8] [1, 4, 4] [1, 1, 1] : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
        memref.copy %subview_9, %subview_10 : memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
      } {mapping = [#gpu.thread<y>, #gpu.thread<x>]}
      scf.yield %arg1 : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    }
    %subview_3 = memref.subview %2[%3, %4, %5] [1, 32, 128] [1, 1, 1] : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    memref.copy %6, %subview_3 : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    return
  }
}

// -----// IR Dump After FoldMemRefAliasOps (fold-memref-alias-ops) //----- //
func.func @matmul_dispatch_0_batch_matmul_8x512x512x1024_f32() {
  %c0 = arith.constant 0 : index
  %cst = arith.constant dense<0.000000e+00> : vector<1x4x4xf32>
  %c32 = arith.constant 32 : index
  %c1024 = arith.constant 1024 : index
  %cst_0 = arith.constant 0.000000e+00 : f32
  %0 = gpu.thread_id  x
  %1 = gpu.thread_id  y
  %2 = gpu.thread_id  z
  %alloc = memref.alloc() : memref<1x32x132xf32, #gpu.address_space<workgroup>>
  %alloc_1 = memref.alloc() : memref<1x32x36xf32, #gpu.address_space<workgroup>>
  %3 = hal.interface.binding.subspan set(0) binding(0) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %3, 64 : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  %4 = hal.interface.binding.subspan set(0) binding(1) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %4, 64 : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  %5 = hal.interface.binding.subspan set(0) binding(2) type(storage_buffer) alignment(64) offset(%c0) : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %5, 64 : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  %workgroup_id_x = hal.interface.workgroup.id[0] : index
  %6 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
  %7 = affine.apply affine_map<()[s0, s1] -> (s1 * 4 + (s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x, %1]
  %8 = affine.apply affine_map<()[s0, s1] -> (s0 * 128 + s1 * 4 - (s0 floordiv 4) * 512)>()[%workgroup_id_x, %0]
  vector.transfer_write %cst, %5[%6, %7, %8] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  scf.for %arg0 = %c0 to %c1024 step %c32 {
    gpu.barrier
    %9 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
    %10 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s2 * 4 + s3 * 32 + (s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512 + s1 floordiv 8)>()[%workgroup_id_x, %0, %1, %2]
    %11 = affine.apply affine_map<()[s0, s1] -> (s0 + s1 * 4 - (s1 floordiv 8) * 32)>()[%arg0, %0]
    %12 = vector.transfer_read %3[%9, %10, %11], %cst_0 {in_bounds = [true, true, true]} : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    %13 = affine.apply affine_map<()[s0, s1, s2] -> (s1 * 4 + s2 * 32 + s0 floordiv 8)>()[%0, %1, %2]
    %14 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 8) * 32)>()[%0]
    vector.transfer_write %12, %alloc_1[%c0, %13, %14] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x36xf32, #gpu.address_space<workgroup>>
    %15 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
    %16 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32)>()[%arg0, %0, %1, %2]
    %17 = affine.apply affine_map<()[s0, s1] -> (s0 * 128 + s1 * 4 - (s0 floordiv 4) * 512 - (s1 floordiv 32) * 128)>()[%workgroup_id_x, %0]
    %18 = vector.transfer_read %4[%15, %16, %17], %cst_0 {in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    %19 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32)>()[%0, %1, %2]
    %20 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
    vector.transfer_write %18, %alloc[%c0, %19, %20] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
    %21 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
    %22 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32 + 8)>()[%arg0, %0, %1, %2]
    %23 = affine.apply affine_map<()[s0, s1] -> (s0 * 128 + s1 * 4 - (s0 floordiv 4) * 512 - (s1 floordiv 32) * 128)>()[%workgroup_id_x, %0]
    %24 = vector.transfer_read %4[%21, %22, %23], %cst_0 {in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    %25 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32 + 8)>()[%0, %1, %2]
    %26 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
    vector.transfer_write %24, %alloc[%c0, %25, %26] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
    %27 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
    %28 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32 + 16)>()[%arg0, %0, %1, %2]
    %29 = affine.apply affine_map<()[s0, s1] -> (s0 * 128 + s1 * 4 - (s0 floordiv 4) * 512 - (s1 floordiv 32) * 128)>()[%workgroup_id_x, %0]
    %30 = vector.transfer_read %4[%27, %28, %29], %cst_0 {in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    %31 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32 + 16)>()[%0, %1, %2]
    %32 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
    vector.transfer_write %30, %alloc[%c0, %31, %32] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
    %33 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
    %34 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32 + 24)>()[%arg0, %0, %1, %2]
    %35 = affine.apply affine_map<()[s0, s1] -> (s0 * 128 + s1 * 4 - (s0 floordiv 4) * 512 - (s1 floordiv 32) * 128)>()[%workgroup_id_x, %0]
    %36 = vector.transfer_read %4[%33, %34, %35], %cst_0 {in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    %37 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32 + 24)>()[%0, %1, %2]
    %38 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
    vector.transfer_write %36, %alloc[%c0, %37, %38] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
    gpu.barrier
    %39 = affine.apply affine_map<()[s0] -> (s0 * 4)>()[%1]
    %40 = vector.transfer_read %alloc_1[%c0, %39, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x32x36xf32, #gpu.address_space<workgroup>>, vector<1x4x32xf32>
    %41 = affine.apply affine_map<()[s0] -> (s0 * 4)>()[%0]
    %42 = vector.transfer_read %alloc[%c0, %c0, %41], %cst_0 {in_bounds = [true, true, true]} : memref<1x32x132xf32, #gpu.address_space<workgroup>>, vector<1x32x4xf32>
    %43 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
    %44 = affine.apply affine_map<()[s0, s1] -> (s1 * 4 + (s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x, %1]
    %45 = affine.apply affine_map<()[s0, s1] -> (s0 * 128 + s1 * 4 - (s0 floordiv 4) * 512)>()[%workgroup_id_x, %0]
    %46 = vector.transfer_read %5[%43, %44, %45], %cst_0 {in_bounds = [true, true, true]} : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x4x4xf32>
    %47 = vector.contract {indexing_maps = [affine_map<(d0, d1, d2, d3) -> (d0, d1, d3)>, affine_map<(d0, d1, d2, d3) -> (d0, d3, d2)>, affine_map<(d0, d1, d2, d3) -> (d0, d1, d2)>], iterator_types = ["parallel", "parallel", "parallel", "reduction"], kind = #vector.kind<add>} %40, %42, %46 : vector<1x4x32xf32>, vector<1x32x4xf32> into vector<1x4x4xf32>
    %48 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
    %49 = affine.apply affine_map<()[s0, s1] -> (s1 * 4 + (s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x, %1]
    %50 = affine.apply affine_map<()[s0, s1] -> (s0 * 128 + s1 * 4 - (s0 floordiv 4) * 512)>()[%workgroup_id_x, %0]
    vector.transfer_write %47, %5[%48, %49, %50] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  }
  return
}


// -----// IR Dump After HoistRedundantVectorTransfers (iree-codegen-hoist-redundant-vector-transfers) //----- //
func.func @matmul_dispatch_0_batch_matmul_8x512x512x1024_f32() {
  %c0 = arith.constant 0 : index
  %cst = arith.constant dense<0.000000e+00> : vector<1x4x4xf32>
  %c32 = arith.constant 32 : index
  %c1024 = arith.constant 1024 : index
  %cst_0 = arith.constant 0.000000e+00 : f32
  %0 = gpu.thread_id  x
  %1 = gpu.thread_id  y
  %2 = gpu.thread_id  z
  %alloc = memref.alloc() : memref<1x32x132xf32, #gpu.address_space<workgroup>>
  %alloc_1 = memref.alloc() : memref<1x32x36xf32, #gpu.address_space<workgroup>>
  %3 = hal.interface.binding.subspan set(0) binding(0) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %3, 64 : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  %4 = hal.interface.binding.subspan set(0) binding(1) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %4, 64 : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  %5 = hal.interface.binding.subspan set(0) binding(2) type(storage_buffer) alignment(64) offset(%c0) : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %5, 64 : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  %workgroup_id_x = hal.interface.workgroup.id[0] : index
  %6 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
  %7 = affine.apply affine_map<()[s0, s1] -> (s1 * 4 + (s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x, %1]
  %8 = affine.apply affine_map<()[s0, s1] -> (s0 * 128 + s1 * 4 - (s0 floordiv 4) * 512)>()[%workgroup_id_x, %0]
  %9 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s2 * 4 + s3 * 32 + (s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512 + s1 floordiv 8)>()[%workgroup_id_x, %0, %1, %2]
  %10 = affine.apply affine_map<()[s0, s1, s2] -> (s1 * 4 + s2 * 32 + s0 floordiv 8)>()[%0, %1, %2]
  %11 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 8) * 32)>()[%0]
  %12 = affine.apply affine_map<()[s0, s1] -> (s0 * 128 + s1 * 4 - (s0 floordiv 4) * 512 - (s1 floordiv 32) * 128)>()[%workgroup_id_x, %0]
  %13 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32)>()[%0, %1, %2]
  %14 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
  %15 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32 + 8)>()[%0, %1, %2]
  %16 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32 + 16)>()[%0, %1, %2]
  %17 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32 + 24)>()[%0, %1, %2]
  %18 = affine.apply affine_map<()[s0] -> (s0 * 4)>()[%1]
  %19 = affine.apply affine_map<()[s0] -> (s0 * 4)>()[%0]
  %20 = scf.for %arg0 = %c0 to %c1024 step %c32 iter_args(%arg1 = %cst) -> (vector<1x4x4xf32>) {
    gpu.barrier
    %21 = affine.apply affine_map<()[s0, s1] -> (s0 + s1 * 4 - (s1 floordiv 8) * 32)>()[%arg0, %0]
    %22 = vector.transfer_read %3[%6, %9, %21], %cst_0 {in_bounds = [true, true, true]} : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    vector.transfer_write %22, %alloc_1[%c0, %10, %11] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x36xf32, #gpu.address_space<workgroup>>
    %23 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32)>()[%arg0, %0, %1, %2]
    %24 = vector.transfer_read %4[%6, %23, %12], %cst_0 {in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    vector.transfer_write %24, %alloc[%c0, %13, %14] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
    %25 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32 + 8)>()[%arg0, %0, %1, %2]
    %26 = vector.transfer_read %4[%6, %25, %12], %cst_0 {in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    vector.transfer_write %26, %alloc[%c0, %15, %14] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
    %27 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32 + 16)>()[%arg0, %0, %1, %2]
    %28 = vector.transfer_read %4[%6, %27, %12], %cst_0 {in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    vector.transfer_write %28, %alloc[%c0, %16, %14] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
    %29 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32 + 24)>()[%arg0, %0, %1, %2]
    %30 = vector.transfer_read %4[%6, %29, %12], %cst_0 {in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    vector.transfer_write %30, %alloc[%c0, %17, %14] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
    gpu.barrier
    %31 = vector.transfer_read %alloc_1[%c0, %18, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x32x36xf32, #gpu.address_space<workgroup>>, vector<1x4x32xf32>
    %32 = vector.transfer_read %alloc[%c0, %c0, %19], %cst_0 {in_bounds = [true, true, true]} : memref<1x32x132xf32, #gpu.address_space<workgroup>>, vector<1x32x4xf32>
    %33 = vector.contract {indexing_maps = [affine_map<(d0, d1, d2, d3) -> (d0, d1, d3)>, affine_map<(d0, d1, d2, d3) -> (d0, d3, d2)>, affine_map<(d0, d1, d2, d3) -> (d0, d1, d2)>], iterator_types = ["parallel", "parallel", "parallel", "reduction"], kind = #vector.kind<add>} %31, %32, %arg1 : vector<1x4x32xf32>, vector<1x32x4xf32> into vector<1x4x4xf32>
    scf.yield %33 : vector<1x4x4xf32>
  }
  vector.transfer_write %20, %5[%6, %7, %8] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  return
}


In [ ]:
// -----// IR Dump After ConvertToDestinationPassingStyle (iree-codegen-convert-to-destination-passing-style) //----- //
func.func @matmul_dispatch_0_batch_matmul_8x512x512x1024_f32() {
  %c32 = arith.constant 32 : index
  %c128 = arith.constant 128 : index
  %c0 = arith.constant 0 : index
  %cst = arith.constant 0.000000e+00 : f32
  %0 = hal.interface.binding.subspan set(0) binding(0) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : !flow.dispatch.tensor<readonly:tensor<8x512x1024xf32>>
  %1 = hal.interface.binding.subspan set(0) binding(1) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : !flow.dispatch.tensor<readonly:tensor<8x1024x512xf32>>
  %2 = hal.interface.binding.subspan set(0) binding(2) type(storage_buffer) alignment(64) offset(%c0) : !flow.dispatch.tensor<writeonly:tensor<8x512x512xf32>>
  %workgroup_id_x = hal.interface.workgroup.id[0] : index
  %3 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
  %4 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x]
  %5 = affine.apply affine_map<()[s0] -> (s0 * 128 - (s0 floordiv 4) * 512)>()[%workgroup_id_x]
  %6 = flow.dispatch.tensor.load %2, offsets = [%3, %4, %5], sizes = [1, %c32, %c128], strides = [1, 1, 1] : !flow.dispatch.tensor<writeonly:tensor<8x512x512xf32>> -> tensor<1x?x?xf32>
  %cast = tensor.cast %6 : tensor<1x?x?xf32> to tensor<1x32x128xf32>
  %workgroup_id_x_0 = hal.interface.workgroup.id[0] : index
  %7 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x_0]
  %8 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x_0]
  %9 = flow.dispatch.tensor.load %0, offsets = [%7, %8, 0], sizes = [1, %c32, 1024], strides = [1, 1, 1] : !flow.dispatch.tensor<readonly:tensor<8x512x1024xf32>> -> tensor<1x?x1024xf32>
  %10 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x_0]
  %11 = affine.apply affine_map<()[s0] -> (s0 * 128 - (s0 floordiv 4) * 512)>()[%workgroup_id_x_0]
  %12 = flow.dispatch.tensor.load %1, offsets = [%10, 0, %11], sizes = [1, 1024, %c128], strides = [1, 1, 1] : !flow.dispatch.tensor<readonly:tensor<8x1024x512xf32>> -> tensor<1x1024x?xf32>
  %13 = linalg.fill {lowering_config = #iree_codegen.lowering_config<tile_sizes = [[1, 32, 128, 32]]>} ins(%cst : f32) outs(%cast : tensor<1x32x128xf32>) -> tensor<1x32x128xf32>
  %cast_1 = tensor.cast %12 : tensor<1x1024x?xf32> to tensor<1x1024x128xf32>
  %cast_2 = tensor.cast %9 : tensor<1x?x1024xf32> to tensor<1x32x1024xf32>
  %14 = linalg.batch_matmul {lowering_config = #iree_codegen.lowering_config<tile_sizes = [[1, 32, 128, 32]]>} ins(%cast_2, %cast_1 : tensor<1x32x1024xf32>, tensor<1x1024x128xf32>) outs(%13 : tensor<1x32x128xf32>) -> tensor<1x32x128xf32>
  %cast_3 = tensor.cast %14 : tensor<1x32x128xf32> to tensor<1x?x?xf32>
  %15 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x_0]
  %16 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x_0]
  %17 = affine.apply affine_map<()[s0] -> (s0 * 128 - (s0 floordiv 4) * 512)>()[%workgroup_id_x_0]
  flow.dispatch.tensor.store %cast_3, %2, offsets = [%15, %16, %17], sizes = [1, %c32, %c128], strides = [1, 1, 1] : tensor<1x?x?xf32> -> !flow.dispatch.tensor<writeonly:tensor<8x512x512xf32>>
  return
}


### Bank conflict optimisations
* Shared memory is arranged in banks (32 banks each of width 4 bytes)
* Each thread in a warp can access shared memory in 
paralll, if and only if non-conflict
* conflict lead to consective access
    * When 2 or more threads in a warp access 4 byte words in 
the same ban, they do it sequentially
    * pad inner dimensions with 16 bytes (4xf32), to shift the next access bank
    * why 4/16? how many threads in one wrap, how they shifts
* use shared memory swizzle for better perf.ke

In [ ]:
// -----// IR Dump After GPUReduceBankConflicts (iree-codegen-gpu-reduce-bank-conflicts) //----- //
func.func @matmul_dispatch_0_batch_matmul_8x512x512x1024_f32() {
  %c0 = arith.constant 0 : index
  %cst = arith.constant dense<0.000000e+00> : vector<1x4x4xf32>
  %c32 = arith.constant 32 : index
  %c1024 = arith.constant 1024 : index
  %cst_0 = arith.constant 0.000000e+00 : f32
  %0 = gpu.thread_id  x
  %1 = gpu.thread_id  y
  %2 = gpu.thread_id  z
  %alloc = memref.alloc() : memref<1x32x132xf32, #gpu.address_space<workgroup>>
  %subview = memref.subview %alloc[0, 0, 0] [1, 32, 128] [1, 1, 1] : memref<1x32x132xf32, #gpu.address_space<workgroup>> to memref<1x32x128xf32, strided<[4224, 132, 1]>, #gpu.address_space<workgroup>>
  %alloc_1 = memref.alloc() : memref<1x32x36xf32, #gpu.address_space<workgroup>>
  %subview_2 = memref.subview %alloc_1[0, 0, 0] [1, 32, 32] [1, 1, 1] : memref<1x32x36xf32, #gpu.address_space<workgroup>> to memref<1x32x32xf32, strided<[1152, 36, 1]>, #gpu.address_space<workgroup>>
  %3 = hal.interface.binding.subspan set(0) binding(0) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %3, 64 : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  %4 = hal.interface.binding.subspan set(0) binding(1) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %4, 64 : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  %5 = hal.interface.binding.subspan set(0) binding(2) type(storage_buffer) alignment(64) offset(%c0) : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %5, 64 : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  %workgroup_id_x = hal.interface.workgroup.id[0] : index
  %6 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
  %7 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x]
  %8 = affine.apply affine_map<()[s0] -> (s0 * 128 - (s0 floordiv 4) * 512)>()[%workgroup_id_x]
  %subview_3 = memref.subview %5[%6, %7, %8] [1, 32, 128] [1, 1, 1] : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  %subview_4 = memref.subview %3[%6, %7, 0] [1, 32, 1024] [1, 1, 1] : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>> to memref<1x32x1024xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  %subview_5 = memref.subview %4[%6, 0, %8] [1, 1024, 128] [1, 1, 1] : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>> to memref<1x1024x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  %9 = affine.apply affine_map<()[s0] -> (s0 * 4)>()[%1]
  %10 = affine.apply affine_map<()[s0] -> (s0 * 4)>()[%0]
  %subview_6 = memref.subview %subview_3[0, %9, %10] [1, 4, 4] [1, 1, 1] : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  vector.transfer_write %cst, %subview_6[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  scf.for %arg0 = %c0 to %c1024 step %c32 {
    %subview_7 = memref.subview %subview_4[0, 0, %arg0] [1, 32, 32] [1, 1, 1] : memref<1x32x1024xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x32xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_8 = memref.subview %subview_5[0, %arg0, 0] [1, 32, 128] [1, 1, 1] : memref<1x1024x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    gpu.barrier
    %11 = affine.apply affine_map<()[s0, s1, s2] -> (s1 * 4 + s2 * 32 + s0 floordiv 8)>()[%0, %1, %2]
    %12 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 8) * 32)>()[%0]
    %subview_9 = memref.subview %subview_7[0, %11, %12] [1, 1, 4] [1, 1, 1] : memref<1x32x32xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x1x4xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_10 = memref.subview %subview_2[0, %11, %12] [1, 1, 4] [1, 1, 1] : memref<1x32x32xf32, strided<[1152, 36, 1]>, #gpu.address_space<workgroup>> to memref<1x1x4xf32, strided<[1152, 36, 1], offset: ?>, #gpu.address_space<workgroup>>
    %13 = vector.transfer_read %subview_9[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x1x4xf32, strided<[524288, 1024, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    vector.transfer_write %13, %subview_10[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x1x4xf32, strided<[1152, 36, 1], offset: ?>, #gpu.address_space<workgroup>>
    %subview_11 = memref.subview %subview_8[0, 0, 0] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_12 = memref.subview %subview[0, 0, 0] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, strided<[4224, 132, 1]>, #gpu.address_space<workgroup>> to memref<1x8x128xf32, strided<[4224, 132, 1]>, #gpu.address_space<workgroup>>
    %14 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32)>()[%0, %1, %2]
    %15 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
    %subview_13 = memref.subview %subview_11[0, %14, %15] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_14 = memref.subview %subview_12[0, %14, %15] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[4224, 132, 1]>, #gpu.address_space<workgroup>> to memref<1x1x4xf32, strided<[4224, 132, 1], offset: ?>, #gpu.address_space<workgroup>>
    %16 = vector.transfer_read %subview_13[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    vector.transfer_write %16, %subview_14[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x1x4xf32, strided<[4224, 132, 1], offset: ?>, #gpu.address_space<workgroup>>
    %subview_15 = memref.subview %subview_8[0, 8, 0] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_16 = memref.subview %subview[0, 8, 0] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, strided<[4224, 132, 1]>, #gpu.address_space<workgroup>> to memref<1x8x128xf32, strided<[4224, 132, 1], offset: 1056>, #gpu.address_space<workgroup>>
    %subview_17 = memref.subview %subview_15[0, %14, %15] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_18 = memref.subview %subview_16[0, %14, %15] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[4224, 132, 1], offset: 1056>, #gpu.address_space<workgroup>> to memref<1x1x4xf32, strided<[4224, 132, 1], offset: ?>, #gpu.address_space<workgroup>>
    %17 = vector.transfer_read %subview_17[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    vector.transfer_write %17, %subview_18[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x1x4xf32, strided<[4224, 132, 1], offset: ?>, #gpu.address_space<workgroup>>
    %subview_19 = memref.subview %subview_8[0, 16, 0] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_20 = memref.subview %subview[0, 16, 0] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, strided<[4224, 132, 1]>, #gpu.address_space<workgroup>> to memref<1x8x128xf32, strided<[4224, 132, 1], offset: 2112>, #gpu.address_space<workgroup>>
    %subview_21 = memref.subview %subview_19[0, %14, %15] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_22 = memref.subview %subview_20[0, %14, %15] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[4224, 132, 1], offset: 2112>, #gpu.address_space<workgroup>> to memref<1x1x4xf32, strided<[4224, 132, 1], offset: ?>, #gpu.address_space<workgroup>>
    %18 = vector.transfer_read %subview_21[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    vector.transfer_write %18, %subview_22[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x1x4xf32, strided<[4224, 132, 1], offset: ?>, #gpu.address_space<workgroup>>
    %subview_23 = memref.subview %subview_8[0, 24, 0] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_24 = memref.subview %subview[0, 24, 0] [1, 8, 128] [1, 1, 1] : memref<1x32x128xf32, strided<[4224, 132, 1]>, #gpu.address_space<workgroup>> to memref<1x8x128xf32, strided<[4224, 132, 1], offset: 3168>, #gpu.address_space<workgroup>>
    %subview_25 = memref.subview %subview_23[0, %14, %15] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %subview_26 = memref.subview %subview_24[0, %14, %15] [1, 1, 4] [1, 1, 1] : memref<1x8x128xf32, strided<[4224, 132, 1], offset: 3168>, #gpu.address_space<workgroup>> to memref<1x1x4xf32, strided<[4224, 132, 1], offset: ?>, #gpu.address_space<workgroup>>
    %19 = vector.transfer_read %subview_25[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x1x4xf32, strided<[524288, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    vector.transfer_write %19, %subview_26[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x1x4xf32, strided<[4224, 132, 1], offset: ?>, #gpu.address_space<workgroup>>
    gpu.barrier
    %20 = affine.apply affine_map<(d0) -> (d0 * 4)>(%1)
    %21 = affine.apply affine_map<(d0) -> (d0 * 4)>(%0)
    %subview_27 = memref.subview %subview_2[0, %20, 0] [1, 4, 32] [1, 1, 1] : memref<1x32x32xf32, strided<[1152, 36, 1]>, #gpu.address_space<workgroup>> to memref<1x4x32xf32, strided<[1152, 36, 1], offset: ?>, #gpu.address_space<workgroup>>
    %subview_28 = memref.subview %subview[0, 0, %21] [1, 32, 4] [1, 1, 1] : memref<1x32x128xf32, strided<[4224, 132, 1]>, #gpu.address_space<workgroup>> to memref<1x32x4xf32, strided<[4224, 132, 1], offset: ?>, #gpu.address_space<workgroup>>
    %subview_29 = memref.subview %subview_3[0, %20, %21] [1, 4, 4] [1, 1, 1] : memref<1x32x128xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>> to memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
    %22 = vector.transfer_read %subview_27[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x4x32xf32, strided<[1152, 36, 1], offset: ?>, #gpu.address_space<workgroup>>, vector<1x4x32xf32>
    %23 = vector.transfer_read %subview_28[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x32x4xf32, strided<[4224, 132, 1], offset: ?>, #gpu.address_space<workgroup>>, vector<1x32x4xf32>
    %24 = vector.transfer_read %subview_29[%c0, %c0, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>, vector<1x4x4xf32>
    %25 = vector.contract {indexing_maps = [affine_map<(d0, d1, d2, d3) -> (d0, d1, d3)>, affine_map<(d0, d1, d2, d3) -> (d0, d3, d2)>, affine_map<(d0, d1, d2, d3) -> (d0, d1, d2)>], iterator_types = ["parallel", "parallel", "parallel", "reduction"], kind = #vector.kind<add>} %22, %23, %24 : vector<1x4x32xf32>, vector<1x32x4xf32> into vector<1x4x4xf32>
    vector.transfer_write %25, %subview_29[%c0, %c0, %c0] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<1x4x4xf32, strided<[262144, 512, 1], offset: ?>, #hal.descriptor_type<storage_buffer>>
  }
  return
}


### GPU tensorisation (GPU only)
purpose: to use tensorcore insts directly.
* linalg.matmul -> vector.contract
* linalg.generic -> vector.transfer_read/write
* sizes are determined by GPU HW spec/isa supports

### GPU pipelining (GPU only)
* Software pipelining at instruction level
* overlapping loads, store and calc, we have single load-store pipelining and double load-store pipeline


In [ ]:
// -----// IR Dump After GPUPipelining (iree-codegen-gpu-pipelining) //----- //
func.func @matmul_dispatch_0_batch_matmul_8x512x512x1024_f32() {
  %c0 = arith.constant 0 : index
  %cst = arith.constant dense<0.000000e+00> : vector<1x4x4xf32>
  %c32 = arith.constant 32 : index
  %c1024 = arith.constant 1024 : index
  %cst_0 = arith.constant 0.000000e+00 : f32
  %0 = gpu.thread_id  x
  %1 = gpu.thread_id  y
  %2 = gpu.thread_id  z
  %alloc = memref.alloc() : memref<1x32x132xf32, #gpu.address_space<workgroup>>
  %alloc_1 = memref.alloc() : memref<1x32x36xf32, #gpu.address_space<workgroup>>
  %3 = hal.interface.binding.subspan set(0) binding(0) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %3, 64 : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>
  %4 = hal.interface.binding.subspan set(0) binding(1) type(storage_buffer) alignment(64) offset(%c0) flags(ReadOnly) : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %4, 64 : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>
  %5 = hal.interface.binding.subspan set(0) binding(2) type(storage_buffer) alignment(64) offset(%c0) : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  memref.assume_alignment %5, 64 : memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  %workgroup_id_x = hal.interface.workgroup.id[0] : index
  %6 = affine.apply affine_map<()[s0] -> ((s0 floordiv 4) floordiv 16)>()[%workgroup_id_x]
  %7 = affine.apply affine_map<()[s0, s1] -> (s1 * 4 + (s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512)>()[%workgroup_id_x, %1]
  %8 = affine.apply affine_map<()[s0, s1] -> (s0 * 128 + s1 * 4 - (s0 floordiv 4) * 512)>()[%workgroup_id_x, %0]
  %9 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s2 * 4 + s3 * 32 + (s0 floordiv 4) * 32 - ((s0 floordiv 4) floordiv 16) * 512 + s1 floordiv 8)>()[%workgroup_id_x, %0, %1, %2]
  %10 = affine.apply affine_map<()[s0, s1, s2] -> (s1 * 4 + s2 * 32 + s0 floordiv 8)>()[%0, %1, %2]
  %11 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 8) * 32)>()[%0]
  %12 = affine.apply affine_map<()[s0, s1] -> (s0 * 128 + s1 * 4 - (s0 floordiv 4) * 512 - (s1 floordiv 32) * 128)>()[%workgroup_id_x, %0]
  %13 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32)>()[%0, %1, %2]
  %14 = affine.apply affine_map<()[s0] -> (s0 * 4 - (s0 floordiv 32) * 128)>()[%0]
  %15 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32 + 8)>()[%0, %1, %2]
  %16 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32 + 16)>()[%0, %1, %2]
  %17 = affine.apply affine_map<()[s0, s1, s2] -> (s1 + s2 * 8 + s0 floordiv 32 + 24)>()[%0, %1, %2]
  %18 = affine.apply affine_map<()[s0] -> (s0 * 4)>()[%1]
  %19 = affine.apply affine_map<()[s0] -> (s0 * 4)>()[%0]
  %c0_2 = arith.constant 0 : index
  %20 = affine.apply affine_map<()[s0, s1] -> (s0 + s1 * 4 - (s1 floordiv 8) * 32)>()[%c0_2, %0]
  %21 = vector.transfer_read %3[%6, %9, %20], %cst_0 {__pipelining_first_stage__, in_bounds = [true, true, true]} : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
  %22 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32)>()[%c0_2, %0, %1, %2]
  %23 = vector.transfer_read %4[%6, %22, %12], %cst_0 {__pipelining_first_stage__, in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
  %24 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32 + 8)>()[%c0_2, %0, %1, %2]
  %25 = vector.transfer_read %4[%6, %24, %12], %cst_0 {__pipelining_first_stage__, in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
  %26 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32 + 16)>()[%c0_2, %0, %1, %2]
  %27 = vector.transfer_read %4[%6, %26, %12], %cst_0 {__pipelining_first_stage__, in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
  %28 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32 + 24)>()[%c0_2, %0, %1, %2]
  %29 = vector.transfer_read %4[%6, %28, %12], %cst_0 {__pipelining_first_stage__, in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
  %c992 = arith.constant 992 : index
  %30:6 = scf.for %arg0 = %c0 to %c992 step %c32 iter_args(%arg1 = %cst, %arg2 = %21, %arg3 = %23, %arg4 = %25, %arg5 = %27, %arg6 = %29) -> (vector<1x4x4xf32>, vector<1x1x4xf32>, vector<1x1x4xf32>, vector<1x1x4xf32>, vector<1x1x4xf32>, vector<1x1x4xf32>) {
    gpu.barrier
    vector.transfer_write %arg2, %alloc_1[%c0, %10, %11] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x36xf32, #gpu.address_space<workgroup>>
    vector.transfer_write %arg3, %alloc[%c0, %13, %14] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
    vector.transfer_write %arg4, %alloc[%c0, %15, %14] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
    vector.transfer_write %arg5, %alloc[%c0, %16, %14] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
    vector.transfer_write %arg6, %alloc[%c0, %17, %14] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
    gpu.barrier
    %34 = vector.transfer_read %alloc_1[%c0, %18, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x32x36xf32, #gpu.address_space<workgroup>>, vector<1x4x32xf32>
    %35 = vector.transfer_read %alloc[%c0, %c0, %19], %cst_0 {in_bounds = [true, true, true]} : memref<1x32x132xf32, #gpu.address_space<workgroup>>, vector<1x32x4xf32>
    %36 = vector.contract {indexing_maps = [affine_map<(d0, d1, d2, d3) -> (d0, d1, d3)>, affine_map<(d0, d1, d2, d3) -> (d0, d3, d2)>, affine_map<(d0, d1, d2, d3) -> (d0, d1, d2)>], iterator_types = ["parallel", "parallel", "parallel", "reduction"], kind = #vector.kind<add>} %34, %35, %arg1 : vector<1x4x32xf32>, vector<1x32x4xf32> into vector<1x4x4xf32>
    %c32_4 = arith.constant 32 : index
    %37 = arith.addi %arg0, %c32_4 : index
    %38 = affine.apply affine_map<()[s0, s1] -> (s0 + s1 * 4 - (s1 floordiv 8) * 32)>()[%37, %0]
    %39 = vector.transfer_read %3[%6, %9, %38], %cst_0 {__pipelining_first_stage__, in_bounds = [true, true, true]} : memref<8x512x1024xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    %c32_5 = arith.constant 32 : index
    %40 = arith.addi %arg0, %c32_5 : index
    %41 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32)>()[%40, %0, %1, %2]
    %42 = vector.transfer_read %4[%6, %41, %12], %cst_0 {__pipelining_first_stage__, in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    %c32_6 = arith.constant 32 : index
    %43 = arith.addi %arg0, %c32_6 : index
    %44 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32 + 8)>()[%43, %0, %1, %2]
    %45 = vector.transfer_read %4[%6, %44, %12], %cst_0 {__pipelining_first_stage__, in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    %c32_7 = arith.constant 32 : index
    %46 = arith.addi %arg0, %c32_7 : index
    %47 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32 + 16)>()[%46, %0, %1, %2]
    %48 = vector.transfer_read %4[%6, %47, %12], %cst_0 {__pipelining_first_stage__, in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    %c32_8 = arith.constant 32 : index
    %49 = arith.addi %arg0, %c32_8 : index
    %50 = affine.apply affine_map<()[s0, s1, s2, s3] -> (s0 + s2 + s3 * 8 + s1 floordiv 32 + 24)>()[%49, %0, %1, %2]
    %51 = vector.transfer_read %4[%6, %50, %12], %cst_0 {__pipelining_first_stage__, in_bounds = [true, true, true]} : memref<8x1024x512xf32, #hal.descriptor_type<storage_buffer>>, vector<1x1x4xf32>
    scf.yield %36, %39, %42, %45, %48, %51 : vector<1x4x4xf32>, vector<1x1x4xf32>, vector<1x1x4xf32>, vector<1x1x4xf32>, vector<1x1x4xf32>, vector<1x1x4xf32>
  }
  %c992_3 = arith.constant 992 : index
  gpu.barrier
  vector.transfer_write %30#1, %alloc_1[%c0, %10, %11] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x36xf32, #gpu.address_space<workgroup>>
  vector.transfer_write %30#2, %alloc[%c0, %13, %14] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
  vector.transfer_write %30#3, %alloc[%c0, %15, %14] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
  vector.transfer_write %30#4, %alloc[%c0, %16, %14] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
  vector.transfer_write %30#5, %alloc[%c0, %17, %14] {in_bounds = [true, true, true]} : vector<1x1x4xf32>, memref<1x32x132xf32, #gpu.address_space<workgroup>>
  gpu.barrier
  %31 = vector.transfer_read %alloc_1[%c0, %18, %c0], %cst_0 {in_bounds = [true, true, true]} : memref<1x32x36xf32, #gpu.address_space<workgroup>>, vector<1x4x32xf32>
  %32 = vector.transfer_read %alloc[%c0, %c0, %19], %cst_0 {in_bounds = [true, true, true]} : memref<1x32x132xf32, #gpu.address_space<workgroup>>, vector<1x32x4xf32>
  %33 = vector.contract {indexing_maps = [affine_map<(d0, d1, d2, d3) -> (d0, d1, d3)>, affine_map<(d0, d1, d2, d3) -> (d0, d3, d2)>, affine_map<(d0, d1, d2, d3) -> (d0, d1, d2)>], iterator_types = ["parallel", "parallel", "parallel", "reduction"], kind = #vector.kind<add>} %31, %32, %30#0 : vector<1x4x32xf32>, vector<1x32x4xf32> into vector<1x4x4xf32>
  vector.transfer_write %33, %5[%6, %7, %8] {in_bounds = [true, true, true]} : vector<1x4x4xf32>, memref<8x512x512xf32, #hal.descriptor_type<storage_buffer>>
  return
}


### Last mile optimisation between HW-specific codegen
* unrolling vector representations
    * unroll to reduce vector length to HW supported scale (like 128 bits for V100)
    * Pre-emptively handles non power of 2 sizes to avoid suboptimal code generation
* vector.contract -> vector.outproduct / innerproduct -> specific vector/tensor insts for HW, like gpu.fma
* to HW-specific BC format, LLVM IR, NVVM/NVPTX, SPIR/SPIR-V

### Convert to NVVM (GPU only)
use GPU RT features, and lowering former dialect to GPU-specific abstraction level
* SM copy -> nvgpu.async_copy/write/wait ...
* gpu.subgroup_mma_load/store_matrix / subgroup_mma_compute

### Differences between CPU and GPU
CPU: less cores, L1 tied to each core, more cache hierarchies, all cached, targeting to scalar and SIMD/vector operations
GPU: more cores, more threads, L1 shared by a set of cores, less cache hierarchies, cache + scratchpad memory, targeting fma-like tensor operations supported by tensor-cores
Bufferisation:
* CPU pipeline performs bufferization late, prefering 
in-place bufferization while taking care to avoid RaW
conflicts
* GPU pipeline bufferizes early and focuses on 
optimizing shared memory copies and reducing
bank confl
Vectorisation:ic
* GPU can do vectorisation earlier, to set constraints that largely simplified latter analysis and optimisationsts

## Workflow of IREE system
top-level    buildIREEVMTransformPassPipeline = buildIREEPrecompileTransformPassPipeline + buildHALTransformPassPipeline
* buildIREEPrecompileTransformPassPipeline
    * [tag] == START
        * iree::hal::creawteAssignTargetDevicesPass
        * extendInputConversionPreprocessingPassPipeline | crateAutoInputConversionPipelinePass
        * buildCommonInputConversionPassPipeline :: iree-common-input-transformation-pipeline
            * createIREEImportPublicPass
            * createImportMLProgramPass
            * createSanitizeModuleNamesPass
            * iree::flow::createConvertMeshToFlowPass
    * [tag] == INPUT
        * iree::abi::buildTransformPassPipeline | iree::tflite::buildTransformPassPipeline :: iree-abi-transformation-pipeline
    * [tag] == ABI
        * buildConstEvalPassPipelineHooks
        * buildPreprocessingPassPipeline
    * [tag] == Preprocessing
        * buildGlobalOptimizationPassPipeline :: iree-global-optimization-transformation-pipeline
    * [tag] == GlobalOptimisation
* buildIREEVMtransformPassPipeline
    * buildIREEPrecompileTransformPassPipeline
    * [tag] == GlobalOptimisation
    * buildFlowTransformPassPipeline :: iree-flow-transformation-pipeline
    * [tag] == Flow
    * buildStreamTransformPassPipeline :: iree-stream-transformation-pipeline
    * [tag] == Stream
    * buildHALTransformPassPipeline (does it affect performances?)
        * ExecutionModel::AsyncInternal => buildHALTransformPassPipeline :: iree-hal-transformation-pipeline
        * ExecutionModel::AsyncExternal => buildHALTransformPassPipeline
        * ExecutionModel::InlineStatic => buildHALInlineStaticTransformPassPipeline
        * ExecutionModel::InlineDynamic => buildHALInlineDynamicTransformPassPipeline
        * workflow:
           * [tag] ==== Device assignment and interface materialization
           * buildHALConfigurationPassPipeline
           * [tag] == ExecutableSources
           * createConfigureExecutablesPass
               * createConfigureTargetExecutableVariantsPass
                   * buildConfigurationPassPipeline => buildLLVMGPUCodegenConfigurationPassPipeline in CUDATargets
                       * addCommonTargetExecutablePreprocessingPasses
                       * createGPUGeneralizeNamedOpsPass
                       * createLLVMGPUSelectLoweringStrategyPass
           * createDumpExecutableSourcesPass :: do dump if required to dump configured executables
           * addExecutableSubstitutionPasses :: handle editing kernels
           * [tag] == ExecutableConfigurations
           * HAL::createTranslateExecutablesPass
           * [tag] == ExecutableTargets
           * HAL::createConvertToHALPass
           * HAL::createLinkExecutablesPass
           * HAL::createResolveExportOrdinalsPass
           * HAL::createMaterializeResourceCachesPass
           * HAL::createMemoizeDeviceQueriesPass
           * createAffineExpandIndexOpsPass
           * createLowerAffinePass
           * createConvertSCFToCFPass
           * createCombineInitializersPass
           * createSerializeExecutablesPass
           * createIPOPass with fixed point iterator pass
    * [tag] == HAL

ine

## Compile stage
IREEVMPipelinePhase
* Start, very start of compilation procedure
* Input, input pre-conversions to feed core IREE
* ABI, adjust program ABI for specified envs
* Preprocessing, stage after preprocessing done
* GlobalOptimization, stage after global optimisation done
* Flow, with flow dialect as control-flow
* Stream, with stream dialect as lower level control-flow close to HWs
* ExecutableSources, stage before hal.executable been configured
* ExecutableConfigurations, stage before hal.executable been translated to targets, and before been configured
* ExecutableTargets, stage after hal.executable translated to target's IR
* HAL, stage after lowers to HAL dialect
* VM, stage after lowers to VM dialect
* END, all end

In [ ]:
LLVMGPUSelectLoweringStrategyPass
  initGPULaunchConfig
    setRootConfig
      setTransformDialectConfig | setXXXConfig | setRootDefaultConfig
        check --iree-codegen-llvmgpu-enable-transform-dialect-jit=1
        matchAndSetTransformStrategy
          matchAndSetXXXXStrategy (Pad or Linalg) :: XXXX = Pad | Reduction | Matmul | BatchMatmul | Convolution
            Matmul :: 
